# Import required libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import joblib

# Load the dataset

In [2]:
dataset = pd.read_csv('Churn_Modelling.csv')


# List of columns to drop
columns_to_drop = ['RowNumber', 'CustomerId', 'Surname']

# Drop the specified columns
dataset = dataset.drop(columns=columns_to_drop)




X = dataset.drop('Exited', axis=1)
y = dataset['Exited']

# Define preprocessing steps

In [3]:
numerical_features = ['CreditScore', 'Age','Tenure','Balance','NumOfProducts',
                     'HasCrCard','IsActiveMember','EstimatedSalary']  
categorical_features = ['Geography', 'Gender']  



# Preprocessing for numerical data
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])



# Preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])



# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])




In [ ]:
numerical_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                      'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
numerical_data = dataset[numerical_features]


# Apply scaling to numerical features
scaler = StandardScaler()
numerical_data_scaled = scaler.fit_transform(numerical_data)


# Categorical features
categorical_features = ['Geography', 'Gender']
categorical_data = dataset[categorical_features]


# Apply one-hot encoding to categorical features
encoder = OneHotEncoder(handle_unknown='ignore')
categorical_data_encoded = encoder.fit_transform(categorical_data).toarray()


# Combine the preprocessed numerical and categorical data
processed_data = np.hstack([numerical_data_scaled, categorical_data_encoded])


# Get the column names for encoded categorical features
encoded_feature_names = encoder.get_feature_names_out(categorical_features)


# Create a final dataframe
final_df = pd.DataFrame(processed_data, columns=numerical_features + list(encoded_feature_names))


# Display the final dataframe
final_df


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_bangalore,Geography_delhi,Geography_mumbai,Gender_Female,Gender_Male
0,-0.326221,0.293517,-1.041760,-1.225848,-0.911583,0.646092,0.970243,0.021886,0.0,1.0,0.0,1.0,0.0
1,-0.440036,0.198164,-1.387538,0.117350,-0.911583,-1.547768,0.970243,0.216534,1.0,0.0,0.0,1.0,0.0
2,-1.536794,0.293517,1.032908,1.333053,2.527057,0.646092,-1.030670,0.240687,0.0,1.0,0.0,1.0,0.0
3,0.501521,0.007457,-1.387538,-1.225848,0.807737,-1.547768,-1.030670,-0.108918,0.0,1.0,0.0,1.0,0.0
4,2.063884,0.388871,-1.041760,0.785728,-0.911583,0.646092,0.970243,-0.365276,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,1.246488,0.007457,-0.004426,-1.225848,0.807737,0.646092,-1.030670,-0.066419,0.0,1.0,0.0,0.0,1.0
9996,-1.391939,-0.373958,1.724464,-0.306379,-0.911583,0.646092,0.970243,0.027988,0.0,1.0,0.0,0.0,1.0
9997,0.604988,-0.278604,0.687130,-1.225848,-0.911583,-1.547768,0.970243,-1.008643,0.0,1.0,0.0,1.0,0.0
9998,1.256835,0.293517,-0.695982,-0.022608,0.807737,0.646092,-1.030670,-0.125231,0.0,0.0,1.0,0.0,1.0


# Define the classification models

In [6]:
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'SVC': SVC(),
    'Random Forest': RandomForestClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
}

# Evaluate models

In [7]:
results = {}
for name, model in models.items():
    clf = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
    results[name] = scores
    print(f'{name}: {scores.mean():.4f} (+/- {scores.std():.4f})')


Logistic Regression: 0.8097 (+/- 0.0050)
Decision Tree: 0.7924 (+/- 0.0060)
SVC: 0.8562 (+/- 0.0062)
Random Forest: 0.8649 (+/- 0.0053)
Gradient Boosting: 0.8642 (+/- 0.0069)
XGBoost: 0.8540 (+/- 0.0040)


## Choose the best one

In [8]:
# Choose the best model
best_model_name = max(results, key=lambda k: results[k].mean())
best_model = models[best_model_name]
print(f'Best model: {best_model_name}')


Best model: Random Forest


# Train the best model on the entire training data

In [9]:
best_clf = Pipeline(steps=[('preprocessor', preprocessor), 
                           ('classifier', best_model)])
best_clf.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['CreditScore', 'Age',
                                                   'Tenure', 'Balance',
                                                   'NumOfProducts', 'HasCrCard',
                                                   'IsActiveMember',
                                                   'EstimatedSalary']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Geography', 'Gender'])])),
                ('classifier', RandomForestClassifier())])

# Evaluate the best model on the test set

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                    random_state=42)
best_clf.fit(X_train, y_train)
y_pred = best_clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      0.97      0.92      1607
           1       0.77      0.45      0.57       393

    accuracy                           0.87      2000
   macro avg       0.82      0.71      0.74      2000
weighted avg       0.86      0.87      0.85      2000



# Save the best model using joblib

In [11]:
joblib.dump(best_clf, 'model.joblib')

['model.joblib']

In [12]:
import joblib
import pandas as pd

# Load the saved model from the joblib file
best_clf = joblib.load('model.joblib')

# Example test data, replace with your actual test data
test_data = {
    'CreditScore': [619, 608, 502],
    'Geography': ['delhi', 'banglore', 'mumbai'],
    'Gender': ['Female', 'Male', 'Female'],
    'Age': [42, 44, 43],
    'Tenure':[10,8,7],
    'Balance':[83807.86,159660.80,0.00],
    'NumOfProducts':[2,3,5],
    'HasCrCard':[0,1,1],
    'IsActiveMember':[1,1,0],
    'EstimatedSalary':[101348.88,112542.58,113931.57]
    
}
X_test = pd.DataFrame(test_data)

# Make predictions
y_pred = best_clf.predict(X_test)
print("Predictions:", y_pred)


Predictions: [0 1 1]
